# Day 3 · 코드 리뷰 Agent

하나의 입력을 8개 차시 동안 확장합니다. 웹사이트 확인이 아니라 코드·명령·test·결과 파일을 직접 다루며, 모든 외부 쓰기는 dry-run과 사람 승인을 먼저 거칩니다.

In [ ]:
from pathlib import Path
import importlib.util, json, subprocess, sys

def find_workspace(start):
    for candidate in [start, *start.parents]:
        if (candidate / "requirements-day1.txt").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("WORKSPACE_ROOT_NOT_FOUND")

ROOT = find_workspace(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print({"workspace": ROOT.name, "python": sys.version.split()[0]})

In [ ]:
# 최초 1회. 없는 핵심 library가 있을 때만 현재 Notebook Kernel에 설치합니다.
required = ["pydantic", "pytest", "langchain_core", "langgraph"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-day1.txt")],
        check=True,
    )
print({"missing_before_install": missing, "environment_ready": True})

# 실제 STT를 실행할 사람만 requirements-stt-optional.txt를 별도로 설치합니다.

In [ ]:
OUT = ROOT / "output/course-labs/day3"
OUT.mkdir(parents=True, exist_ok=True)

def save_json(name, payload):
    path = OUT / name
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print({"saved": str(path.relative_to(ROOT))})
    return path

def run_command(*args, cwd=ROOT):
    completed = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    display_args = list(args)
    if display_args and display_args[0] == sys.executable:
        display_args[0] = "python"
    result = {
        "command": " ".join(display_args),
        "returncode": completed.returncode,
        "stdout_tail": completed.stdout.strip().splitlines()[-5:],
        "stderr_tail": completed.stderr.strip().splitlines()[-5:],
    }
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result

## 1차시 · Review Rubric과 Severity

finding은 문체가 아니라 사용자 영향·재현 조건·변경 라인·가장 작은 교정으로 작성합니다.

In [ ]:
review_rubric = {
    "required_fields": ["path", "line", "severity", "title", "body", "evidence", "suggestion", "confidence", "rule_id"],
    "severity": {"P0": "즉시 악용·데이터 손실", "P1": "외부 상태·핵심 기능 위험", "P2": "복구·운영 품질 저하", "P3": "국소적 개선"},
    "excluded": ["style_only", "unrelated_file", "invented_runtime_result"],
}
save_json("01_review_rubric.json", review_rubric)
review_rubric

## 2차시 · Unified Diff와 Line Mapping

`+++` target과 `@@` hunk를 읽고 추가된 new line만 복원합니다.

In [ ]:
from src.course_services.review_service import parse_unified_diff

diff_path = ROOT / "data/day3_review_cases/unsafe_pr.diff"
diff_text = diff_path.read_text(encoding="utf-8")
parsed = parse_unified_diff(diff_text)
parsed_result = {
    "changed_paths": parsed.changed_paths,
    "added_lines": [{"path": line.path, "line": line.line, "text": line.text} for line in parsed.added_lines],
}
save_json("02_parsed_hunks.json", parsed_result)
[(line.line, line.text) for line in parsed.added_lines]

## 3차시 · Review Context Pack

전체 저장소가 아니라 변경 path·관련 test 후보·검토 초점만 모델에 전달합니다.

In [ ]:
from src.course_services.review_service import build_context_pack

context_pack = build_context_pack(parsed)
assert "style_only" in context_pack["excluded_focus"]
save_json("03_context_pack.json", context_pack)
context_pack

## 4차시 · ReviewFinding Contract

line·severity·confidence·자동 게시 금지를 Pydantic 계약으로 검증합니다.

In [ ]:
from pydantic import ValidationError
from src.course_services.contracts import ReviewReport
from src.course_services.review_service import run_review_service

review_report = run_review_service(diff_text)
validated_report = ReviewReport.model_validate(review_report)
try:
    ReviewReport(status="SUCCESS", automatic_publish=True)
    contract_boundary = {"status": "UNEXPECTED_SUCCESS"}
except ValidationError as exc:
    contract_boundary = {"status": "EXPECTED_FAILURE", "error_code": str(exc).splitlines()[0]}
save_json("04_review_contract.json", {"normal": validated_report.model_dump(mode="json"), "boundary": contract_boundary})
{"finding_count": len(validated_report.findings), "boundary": contract_boundary}

## 5차시 · Deterministic Baseline

LLM 전에 재현 가능한 세 규칙으로 eval·외부 쓰기·broad exception을 찾습니다.

In [ ]:
baseline = run_review_service(diff_text)
rule_ids = [finding["rule_id"] for finding in baseline["findings"]]
empty_diff = run_review_service("")
assert len(rule_ids) == 3
assert empty_diff["error_code"] == "EMPTY_DIFF"
save_json("05_baseline_review.json", {"normal": baseline, "boundary": empty_diff})
rule_ids

## 6차시 · Static·Test·LLM Hybrid Review

결정론 finding과 실제 test 증거를 먼저 묶고, LLM 의견은 선택 adapter로 분리합니다.

In [ ]:
test_evidence = run_command(sys.executable, "-m", "pytest", "-q", "tests/test_course_services.py", "-k", "unified_diff or maps_findings")
hybrid_review = {
    "static_findings": baseline["findings"],
    "test_evidence": test_evidence,
    "llm_adapter": {"requested": False, "reason": "DETERMINISTIC_BASELINE_FIRST"},
    "ready_for_evaluation": test_evidence["returncode"] == 0,
}
assert hybrid_review["ready_for_evaluation"] is True
save_json("06_hybrid_review.json", hybrid_review)
{"finding_count": len(hybrid_review["static_findings"]), "test_returncode": test_evidence["returncode"]}

## 7차시 · Precision·Recall·F1

Golden finding과 현재 결과를 path·line·rule ID로 비교합니다.

In [ ]:
from src.course_services.eval_service import evaluate_review_findings, release_gate

expected = json.loads((ROOT / "data/day5_eval/golden_review_findings.json").read_text(encoding="utf-8"))
review_metrics = evaluate_review_findings(baseline["findings"], expected)
review_gate = release_gate(review_metrics=review_metrics, safety_passed=True, latency_seconds=0.2)
assert review_gate["decision"] == "READY"
save_json("07_review_eval.json", {"metrics": review_metrics, "release_gate": review_gate})
{"precision": review_metrics["precision"], "recall": review_metrics["recall"], "f1": review_metrics["f1"]}

## 8차시 · Codex Harness와 회귀 방지

Codex 작업을 목표·허용 경로·test·금지 행동으로 제한하고, scope·test·diff·secret을 사람 merge 전에 검사합니다.

In [ ]:
from src.course_services.codex_harness import CodexTaskSpec, assess_codex_run, render_codex_task
from src.course_services.course_demo import build_course_demo

spec = CodexTaskSpec(
    objective="review rule 하나와 정상·실패 test를 추가한다.",
    allowed_paths=("src/course_services", "tests"),
    acceptance_tests=("python -m pytest -q tests/test_course_services.py",),
)
ready_assessment = assess_codex_run(
    spec,
    changed_paths=["src/course_services/review_service.py", "tests/test_course_services.py"],
    executed_tests={"python -m pytest -q tests/test_course_services.py": True},
    diff_reviewed=True, secrets_detected=False,
)
hold_assessment = assess_codex_run(
    spec, changed_paths=[".env"], executed_tests={}, diff_reviewed=False, secrets_detected=True,
)
scorecard = build_course_demo(3, workspace_root=ROOT)
day3_result = {
    "codex_task": render_codex_task(spec), "ready_assessment": ready_assessment,
    "hold_assessment": hold_assessment, "scorecard": scorecard,
}
assert ready_assessment["decision"] == "READY_FOR_HUMAN_MERGE"
assert hold_assessment["decision"] == "HOLD"
save_json("08_day3_report.json", day3_result)
{"decision": scorecard["decision"], "metrics": scorecard["metrics"]}

## 완료 확인

- Day 3의 1~8차시 결과 파일을 확인했습니다.
- 정상 경로와 가장 중요한 실패 경로를 모두 실행했습니다.
- 외부 쓰기와 자동 메일이 기본값 `false`임을 확인했습니다.
- Codex·Claude Code 결과는 test와 diff를 사람이 검토한 뒤에만 반영합니다.